# Notebook 3 — Clean, Transform, and Build the Analytical Base Dataset

## Section 1: Setup and load raw data

In this section we:
- start a Spark session
- load the raw Video_Games reviews and metadata files
- confirm row counts and key columns before cleaning

**Note:** The following environment setup is needed on Windows machine so Spark can write Parquet successfully.

In [1]:
import os

# 1. Point to the base hadoop folder
os.environ['HADOOP_HOME'] = r'C:\hadoop'

# 2. Add the bin folder to the system path so Spark finds hadoop.dll
os.environ['PATH'] = os.environ['PATH'] + ';' + r'C:\hadoop\bin'

In [2]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
REVIEWS_PATH = Path("../data/raw/amazon_reviews_2023/video_games/reviews.jsonl")
METADATA_PATH = Path("../data/raw/amazon_reviews_2023/video_games/metadata.jsonl")

assert REVIEWS_PATH.exists(), f"Missing file: {REVIEWS_PATH}"
assert METADATA_PATH.exists(), f"Missing file: {METADATA_PATH}"

spark = (
    SparkSession.builder
    .appName("Notebook_3_Prepare_Amazon_Video_Games")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.driver.memory", "3g")
    .config("spark.memory.fraction", "0.8")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Reviews path:  {REVIEWS_PATH}")
print(f"Metadata path: {METADATA_PATH}")

Spark version: 4.1.1
Reviews path:  ..\data\raw\amazon_reviews_2023\video_games\reviews.jsonl
Metadata path: ..\data\raw\amazon_reviews_2023\video_games\metadata.jsonl


In [4]:
reviews_raw = spark.read.json(str(REVIEWS_PATH))
metadata_raw = spark.read.json(str(METADATA_PATH))

print("=== Row counts ===")
print(f"Reviews rows:  {reviews_raw.count():,}")
print(f"Metadata rows: {metadata_raw.count():,}")

=== Row counts ===
Reviews rows:  4,624,615
Metadata rows: 137,269


In [5]:
required_review_cols = [
    "rating",
    "title",
    "text",
    "parent_asin",
    "user_id",
    "timestamp",
    "verified_purchase",
    "helpful_vote",
]

required_metadata_cols = [
    "parent_asin",
    "title",
    "main_category",
    "categories",
    "price",
    "store",
]

missing_review_cols = [c for c in required_review_cols if c not in reviews_raw.columns]
missing_metadata_cols = [c for c in required_metadata_cols if c not in metadata_raw.columns]

print("=== Column check ===")
print("Missing review columns:", missing_review_cols)
print("Missing metadata columns:", missing_metadata_cols)

assert not missing_review_cols, f"Missing review columns: {missing_review_cols}"
assert not missing_metadata_cols, f"Missing metadata columns: {missing_metadata_cols}"

=== Column check ===
Missing review columns: []
Missing metadata columns: []


## Section 2: Standardize reviews into a reusable base table

In this section we:
- keep the core review columns needed for downstream analysis
- rename ambiguous fields for clarity
- convert the raw millisecond timestamp into Spark timestamp/date fields
- run a compact validation check

We do not drop rows yet.

In [6]:
reviews_base = (
    reviews_raw
    .select(
        F.col("parent_asin"),
        F.col("user_id"),
        F.col("rating").cast("double").alias("rating"),
        F.trim(F.col("title")).alias("review_title"),
        F.trim(F.col("text")).alias("review_text"),
        F.col("verified_purchase").cast("boolean").alias("verified_purchase"),
        F.col("helpful_vote").cast("long").alias("helpful_vote"),
        F.col("timestamp").cast("long").alias("review_timestamp_ms")
    )
    .withColumn("review_ts", F.expr("timestamp_millis(review_timestamp_ms)"))
    .withColumn("review_date", F.to_date("review_ts"))
    .withColumn("review_year", F.year("review_ts"))
    .withColumn("review_month", F.month("review_ts"))
)

In [7]:
reviews_base.printSchema()

root
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- review_title: string (nullable = true)
 |-- review_text: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- review_timestamp_ms: long (nullable = true)
 |-- review_ts: timestamp (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_month: integer (nullable = true)



In [8]:
reviews_base_summary = reviews_base.agg(
    F.count("*").alias("rows"),
    F.min("review_date").alias("min_review_date"),
    F.max("review_date").alias("max_review_date"),
    F.sum(F.when(F.col("rating").isNull(), 1).otherwise(0)).alias("null_rating"),
    F.sum(F.when(F.col("parent_asin").isNull(), 1).otherwise(0)).alias("null_parent_asin"),
    F.sum(F.when(F.col("user_id").isNull(), 1).otherwise(0)).alias("null_user_id"),
    F.sum(F.when(F.col("review_ts").isNull(), 1).otherwise(0)).alias("null_review_ts")
)

reviews_base_summary.show(truncate=False)

+-------+---------------+---------------+-----------+----------------+------------+--------------+
|rows   |min_review_date|max_review_date|null_rating|null_parent_asin|null_user_id|null_review_ts|
+-------+---------------+---------------+-----------+----------------+------------+--------------+
|4624615|1998-11-17     |2023-09-12     |0          |0               |0           |0             |
+-------+---------------+---------------+-----------+----------------+------------+--------------+



## Section 3: Standardize metadata into a reusable base table

In this section we:
- keep the core metadata columns needed downstream
- rename the product title for clarity
- run a compact validation check

In [9]:
metadata_base = (
    metadata_raw
    .select(
        F.col("parent_asin"),
        F.trim(F.col("title")).alias("product_title"),
        F.trim(F.col("main_category")).alias("main_category"),
        F.col("categories"),
        F.trim(F.col("store")).alias("store"),
        F.col("price")
    )
)

In [10]:
metadata_base.printSchema()

root
 |-- parent_asin: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- store: string (nullable = true)
 |-- price: string (nullable = true)



In [11]:
metadata_base_summary = metadata_base.agg(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("parent_asin").isNull(), 1).otherwise(0)).alias("null_parent_asin"),
    F.sum(F.when(F.col("product_title").isNull(), 1).otherwise(0)).alias("null_product_title"),
    F.sum(F.when(F.col("categories").isNull(), 1).otherwise(0)).alias("null_categories"),
    F.sum(F.when(F.col("price").isNull(), 1).otherwise(0)).alias("null_price")
)

metadata_base_summary.show(truncate=False)

+------+----------------+------------------+---------------+----------+
|rows  |null_parent_asin|null_product_title|null_categories|null_price|
+------+----------------+------------------+---------------+----------+
|137269|0               |0                 |0              |0         |
+------+----------------+------------------+---------------+----------+



## Section 4: Clean selected metadata fields

In this section we:
- convert `price` into a numeric field
- clean the `categories` array
- add a few simple helper columns for later use

In [12]:
metadata_clean = (
    metadata_base
    .withColumn(
        "price_value",
        F.expr("try_cast(nullif(trim(price), '') as double)")
    )
    .withColumn(
        "categories_clean",
        F.expr("filter(transform(categories, x -> trim(x)), x -> x is not null and x != '')")
    )
    .withColumn("category_count", F.size("categories_clean"))
    .withColumn(
        "store_clean",
        F.when(F.trim(F.col("store")) == "", None).otherwise(F.trim(F.col("store")))
    )
)

In [13]:
metadata_clean.select(
    "parent_asin",
    "product_title",
    "price",
    "price_value",
    "categories_clean",
    "category_count",
    "store_clean"
).printSchema()

root
 |-- parent_asin: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- price: string (nullable = true)
 |-- price_value: double (nullable = true)
 |-- categories_clean: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- category_count: integer (nullable = true)
 |-- store_clean: string (nullable = true)



In [14]:
metadata_clean_summary = metadata_clean.agg(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("price_value").isNull(), 1).otherwise(0)).alias("null_price_value"),
    F.sum(F.when(F.col("category_count") == 0, 1).otherwise(0)).alias("empty_categories_clean"),
    F.sum(F.when(F.col("store_clean").isNull(), 1).otherwise(0)).alias("null_store_clean")
)

metadata_clean_summary.show(truncate=False)

+------+----------------+----------------------+----------------+
|rows  |null_price_value|empty_categories_clean|null_store_clean|
+------+----------------+----------------------+----------------+
|137269|75297           |12637                 |4361            |
+------+----------------+----------------------+----------------+



## Section 5: Join reviews and metadata

In this section we:
- join the cleaned reviews and metadata tables on `parent_asin`
- keep a compact set of columns for downstream analysis
- validate that the join preserved all review rows

In [15]:
base_df = (
    reviews_base
    .join(
        metadata_clean.select(
            "parent_asin",
            "product_title",
            "main_category",
            "categories_clean",
            "category_count",
            "store_clean",
            "price_value"
        ),
        on="parent_asin",
        how="left"
    )
)

In [16]:
base_df_summary = base_df.agg(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("product_title").isNull(), 1).otherwise(0)).alias("null_product_title"),
    F.sum(F.when(F.col("categories_clean").isNull(), 1).otherwise(0)).alias("null_categories_clean"),
    F.sum(F.when(F.col("price_value").isNull(), 1).otherwise(0)).alias("null_price_value")
)

base_df_summary.show(truncate=False)

+-------+------------------+---------------------+----------------+
|rows   |null_product_title|null_categories_clean|null_price_value|
+-------+------------------+---------------------+----------------+
|4624615|0                 |0                    |1396824         |
+-------+------------------+---------------------+----------------+



## Section 6: Add a few reusable helper columns

In this section we add a small number of helper columns that will be useful in later analysis.

In [17]:
base_df = (
    base_df
    .withColumn("review_year_month", F.date_trunc("month", F.col("review_ts")))
    .withColumn("review_text_length", F.length(F.col("review_text")))
    .withColumn("has_price", F.col("price_value").isNotNull())
    .withColumn("has_categories", F.col("category_count") > 0)
)

In [18]:
base_df.select(
    "review_year_month",
    "review_text_length",
    "has_price",
    "has_categories"
).printSchema()

root
 |-- review_year_month: timestamp (nullable = true)
 |-- review_text_length: integer (nullable = true)
 |-- has_price: boolean (nullable = false)
 |-- has_categories: boolean (nullable = true)



In [19]:
base_df_helper_summary = base_df.agg(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("review_text_length").isNull(), 1).otherwise(0)).alias("null_review_text_length"),
    F.sum(F.when(F.col("has_price"), 1).otherwise(0)).alias("rows_with_price"),
    F.sum(F.when(F.col("has_categories"), 1).otherwise(0)).alias("rows_with_categories")
)

base_df_helper_summary.show(truncate=False)

+-------+-----------------------+---------------+--------------------+
|rows   |null_review_text_length|rows_with_price|rows_with_categories|
+-------+-----------------------+---------------+--------------------+
|4624615|0                      |3227791        |4486609             |
+-------+-----------------------+---------------+--------------------+



## Section 7: Finalize the analytical base table

In this section we:
- select the final columns for the analytical base dataset
- keep the table compact and consistent for downstream notebooks
- run one small validation check

In [20]:
analytical_base_df = base_df.select(
    "parent_asin",
    "user_id",
    "product_title",
    "main_category",
    "categories_clean",
    "category_count",
    "store_clean",
    "price_value",
    "rating",
    "verified_purchase",
    "helpful_vote",
    "review_text_length",
    "review_timestamp_ms",
    "review_ts",
    "review_date",
    "review_year",
    "review_month",
    "review_year_month",
    "has_price",
    "has_categories"
)

In [21]:
analytical_base_df.printSchema()

root
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- categories_clean: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- category_count: integer (nullable = true)
 |-- store_clean: string (nullable = true)
 |-- price_value: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- review_text_length: integer (nullable = true)
 |-- review_timestamp_ms: long (nullable = true)
 |-- review_ts: timestamp (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_month: integer (nullable = true)
 |-- review_year_month: timestamp (nullable = true)
 |-- has_price: boolean (nullable = false)
 |-- has_categories: boolean (nullable = true)



In [22]:
analytical_base_summary = analytical_base_df.agg(
    F.count("*").alias("rows"),
    F.countDistinct("parent_asin").alias("distinct_parent_asin"),
    F.countDistinct("user_id").alias("distinct_user_id")
)

analytical_base_summary.show(truncate=False)

+-------+--------------------+----------------+
|rows   |distinct_parent_asin|distinct_user_id|
+-------+--------------------+----------------+
|4624615|137249              |2766656         |
+-------+--------------------+----------------+



## Section 8: Save the analytical base dataset

In this section we:
- write the final analytical base dataset to Parquet
- read it back once to confirm the save worked

In [23]:
OUTPUT_PATH = str(Path("../data/processed/video_games_analytical_base").resolve())

print(f"Output path: {OUTPUT_PATH}")

(
    analytical_base_df
    .write
    .mode("overwrite")
    .parquet(OUTPUT_PATH)
)

print("Write completed.")

Output path: C:\Users\olive\git\HAMK\BigDataAnalytics\Project\big-data-video-games-reviews\data\processed\video_games_analytical_base
Write completed.


In [24]:
analytical_base_saved = spark.read.parquet(OUTPUT_PATH)

analytical_base_saved.agg(
    F.count("*").alias("rows"),
    F.countDistinct("parent_asin").alias("distinct_parent_asin"),
    F.countDistinct("user_id").alias("distinct_user_id")
).show(truncate=False)

+-------+--------------------+----------------+
|rows   |distinct_parent_asin|distinct_user_id|
+-------+--------------------+----------------+
|4624615|137249              |2766656         |
+-------+--------------------+----------------+



## Section 9: Summary

In this notebook we created a cleaned analytical base dataset by:
- standardizing review fields
- standardizing and cleaning metadata
- joining both datasets on `parent_asin`
- adding a few reusable helper columns
- saving the final dataset to Parquet for later notebooks